# APEX Optimizer Tutorial: Math Problem Solving

This tutorial demonstrates how to use **APEX** (Analysis-based Prompt Engineering eXpert) to improve GPT-5 Mini's performance on AIME math problems through systematic prompt optimization.

APEX analyzes failures, recognizes success patterns, generates hypotheses, and validates improvements empirically.

## Configuration

All modifiable parameters in one place for easy adjustment:

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Pydantic serializer warnings:",
    category=UserWarning,
    module="pydantic.main",
)

In [ ]:
from dspy.teleprompt.apex.litellm_session_pool import set_pool_size_for_litellm_session

# API Configuration
api_key = 'sk-12345'# Will prompt if not set
base_url = 'https://nexus-master.lmndstaging.com'

# Student Model Configuration (model being optimized)
student_model = "litellm_proxy/openai/gpt-5-mini"
student_base_url = base_url  # Optional custom API endpoint
student_temperature = 0.0  # Deterministic for math
student_reasoning_effort = 'minimal'

# Analysis Model Configuration (for failure analysis and hypotheses)
# analysis_model = "litellm_proxy/openai/gpt-5"
analysis_model = "litellm_proxy/vertex_ai/gemini-2.5-pro"
analysis_base_url = base_url  # Optional custom API endpoint
analysis_temperature = 1.0  # Creative for hypothesis generation
analysis_reasoning_effort = None#'minimal'

# APEX Optimization Settings
max_iterations = 50
num_hypotheses = 1
train_sample_size = 10
success_threshold = 1.0
convergence_patience = 10
num_threads = 50
seed = 42
verbosity = "detailed"

# MLflow Tracking (Optional)
use_mlflow = True
mlflow_tracking_uri = "http://localhost:5005"
mlflow_experiment_name = "APEX-AIME-Math"

set_pool_size_for_litellm_session(pool_size=num_threads)

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

Import dependencies and configure language models:

In [3]:
import os
import dspy
from dspy.adapters import JSONAdapter

if api_key is None:
    api_key = os.getenv("OPENAI_API_KEY") or input("Enter your OpenAI API key: ")

# Configure student model
student_kwargs = {
    "model": student_model,
    "api_key": api_key,
    "temperature": student_temperature,
}
if student_base_url:
    student_kwargs["base_url"] = student_base_url
if student_reasoning_effort:
    student_kwargs["reasoning_effort"] = student_reasoning_effort

student_lm = dspy.LM(**student_kwargs)

# Configure analysis model
analysis_kwargs = {
    "model": analysis_model,
    "api_key": api_key,
    "temperature": analysis_temperature,
}
if analysis_base_url:
    analysis_kwargs["base_url"] = analysis_base_url
if analysis_reasoning_effort:
    analysis_kwargs["reasoning_effort"] = analysis_reasoning_effort

analysis_lm = dspy.LM(**analysis_kwargs)

analysis_adapter = JSONAdapter()
hypothesis_adapter = JSONAdapter()

dspy.configure(lm=student_lm)
dspy.settings.configure(num_threads=num_threads)

## Dataset

Load AIME problems (American Invitational Mathematics Examination):

In [4]:
from datasets import load_dataset
import random

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            "solution": x['solution'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

train_set, val_set, test_set = init_dataset()

print(f"Training: {len(train_set)} | Validation: {len(val_set)} | Test: {len(test_set)}")

Training: 45 | Validation: 45 | Test: 150


Example problem:

In [5]:
print("Problem:", train_set[0]['problem'])
print("\nAnswer:", train_set[0]['answer'])

Problem: In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.

Answer: 242


## Program

Define a Chain of Thought program:

In [6]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

## Metrics

Define evaluation metrics:

In [7]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

In [8]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = (
            f"The final answer must be a valid integer. "
            f"You responded with '{prediction.answer}', which couldn't be parsed. "
            f"The correct answer is '{correct_answer}'."
        )
        
        if written_solution:
            feedback_text += f" Here's the full solution:\n{written_solution}"
        
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)
    
    if score == 1:
        feedback_text = f"Correct! The answer is '{correct_answer}'."
    else:
        feedback_text = f"Incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += f" Here's the full solution:\n{written_solution}"

    return dspy.Prediction(score=score, feedback=feedback_text)

## Baseline Evaluation

Evaluate the unoptimized program:

In [9]:
eval_kwargs = dict(
    num_threads=num_threads,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
)

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    **eval_kwargs,
)

print("Evaluating baseline...")
baseline_result = evaluate(program)

print(f"\nBaseline Performance: {baseline_result.score / 100.:.1%}")

Evaluating baseline...
Average Metric: 80.00 / 150 (53.3%): 100%|██████████| 150/150 [00:01<00:00, 126.47it/s]

2025/10/18 16:35:07 INFO dspy.evaluate.evaluate: Average Metric: 80 / 150 (53.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b = b+7 and 97_b = 9b+7. We need b+7 to divide 9b+...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"Set up affine coordinates with A=(0,0), B=(1,0), C=(0,1). Points o...",441,✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We must count assignments of 3 labeled flavors (C, V, S) to 9 dist...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer solutions (x,y) in [-100,100] satisfying 12x^2 - x...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Divisibility by 22 means divisible by 2 and 11. Units digit must b...,279,✔️ [1]



Baseline Performance: 53.3%


## APEX Optimization

Optimize the program with APEX:

In [ ]:
from tqdm.contrib.logging import logging_redirect_tqdm
from dspy.teleprompt.apex import APEX

optimizer = APEX(
    metric=metric_with_feedback,
    analysis_lm=analysis_lm,
    hypothesis_lm=analysis_lm,
    analysis_adapter=analysis_adapter,
    hypothesis_adapter=hypothesis_adapter,
    max_iterations=max_iterations,
    num_hypotheses=num_hypotheses,
    num_eval_runs=1,
    train_sample=train_sample_size,
    success_threshold=success_threshold,
    convergence_patience=convergence_patience,
    num_threads=num_threads,
    verbosity=verbosity,
    seed=seed,
    use_mlflow=use_mlflow,
    mlflow_tracking_uri=mlflow_tracking_uri,
    mlflow_experiment_name=mlflow_experiment_name,
)

print("Starting optimization...")

with logging_redirect_tqdm():
    optimized_program = optimizer.compile(
        student=program,
        trainset=train_set,
        valset=val_set,
    )

print("\nOptimization complete!")

2025/10/18 16:35:07 INFO dspy.teleprompt.apex.apex: APEX: MLflow tracking enabled


Starting optimization...


2025/10/18 16:35:07 INFO dspy.teleprompt.apex.apex: APEX: running with num_threads=50
2025/10/18 16:35:07 INFO dspy.teleprompt.apex.apex: APEX: Configuration - max_iterations=50, num_hypotheses=10, success_threshold=1.00, convergence_patience=5
2025/10/18 16:35:07 INFO dspy.teleprompt.apex.apex: APEX: Using seed=42 for reproducibility
2025/10/18 16:35:07 INFO dspy.teleprompt.apex.apex: APEX: Evaluating initial baseline on validation set


Processed 45 / 45 examples: 100%|██████████| 45/45 [00:02<00:00, 19.65it/s]

2025/10/18 16:35:09 INFO dspy.teleprompt.apex.apex: APEX: Initial baseline score=0.5111


...

Processed 31 / 45 examples:  69%|██████▉   | 31/45 [01:11<00:32,  2.30s/it]

2025/10/18 17:07:49 INFO dspy.teleprompt.apex.apex: APEX: ✓ Optimization complete | 4 iterations | Reason: interrupted
2025/10/18 17:07:49 INFO dspy.teleprompt.apex.apex: APEX: Final score: 0.6444 (+0.1333 from baseline 0.5111)
2025/10/18 17:07:49 INFO dspy.teleprompt.apex.apex: APEX: Summary - evaluated 32 candidates from 28 hypotheses
2025/10/18 17:07:49 INFO dspy.teleprompt.apex.apex: APEX: Best score trajectory across iterations: [0.5333333333333333, 0.6222222222222222, 0.6444444444444445, 0.6444444444444445]



🏃 View run valuable-horse-760 at: http://localhost:5005/#/experiments/1/runs/4b2c43205ab4404b8af3b2a6853bf2c3
🧪 View experiment at: http://localhost:5005/#/experiments/1

╔═══════════════════════════════════════════════════════════════╗
║                    APEX Optimization Summary                     ║
╠═══════════════════════════════════════════════════════════════╣
║ Total Iterations: 4                                               ║
║ Hypotheses Tested: 28                                             ║
║ Candidates Evaluated: 32                                          ║
╠═══════════════════════════════════════════════════════════════╣
║ Initial Score: 0.5111                                             ║
║ Final Score: 0.6444                                               ║
║ Improvement: +0.1333                                              ║
╠═══════════════════════════════════════════════════════════════╣
║ Iter │ Train F/S │ Hypotheses │ Best Score │ Δ from prev ║
╟──────┼──────

[Trace(trace_id=tr-db7ce84e18361093285472b3db6196bb), Trace(trace_id=tr-ca6ee35375cdc975c5b04ae58212c5ee), Trace(trace_id=tr-20d1116f9e39680fb2b01046e7f74024), Trace(trace_id=tr-e6e8e1f0547b0c53f50bbabfc1f104d8), Trace(trace_id=tr-382f328553ecf9965dfa41d297b16946), Trace(trace_id=tr-5d5fe8febb8e963e86f582236dc38ac2), Trace(trace_id=tr-fe631a675f50d1c64794eb5c1b886068), Trace(trace_id=tr-b8676c8d71933c558d2b527c6bb832ef), Trace(trace_id=tr-c3ac905bbc1a9dab8e61d39eaf528fe8), Trace(trace_id=tr-f1898aee0b1c061c36c1ebd99829d9f4)]

2025/10/18 17:09:27 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'Cannot store OpenTelemetry spans: No Experiment with id=1 exists'}
2025/10/18 17:09:27 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'Cannot store OpenTelemetry spans: No Experiment with id=1 exists'}
2025/10/18 17:09:27 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'Cannot store OpenTelemetry spans: No Experiment with id=1 exists'}
2025/10/18 17:09:27 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'Cannot store OpenTelemetry spans: No Experiment with id=1 exists'}
2025/10/18 17:09:27 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'Cannot store OpenTelemetry spans: No Experiment

Inspect the optimized prompt:

In [ ]:
print("Optimized Prompt:")
print("=" * 50)
print(optimized_program.predict.signature.instructions)
print("=" * 50)

Optimized Prompt:
Carefully analyze and solve the mathematical problem provided. Structure your reasoning and provide a final answer.

Follow this methodology for your solution:
1.  **Analyze and Reframe**: Identify the type of problem (e.g., algebra, number theory, geometry, combinatorics). Note the key constraints, variables, and the objective. If possible, reframe the problem into a standard mathematical form.
2.  **Decompose and Strategize**: Break the problem down into smaller, manageable steps. Apply the principle of **Simplicity and Verification First**. Consider potential solution strategies and key principles:
    - **Prioritize Simple Paths:** Always begin by visualizing the problem and searching for the simplest, most elegant solution. Before committing to a complex method (e.g., extensive case analysis, coordinate geometry, brute-force enumeration), double-check if a simpler approach exists (e.g., using symmetry, finding an invariant, applying a core theorem).
    - **Verif

## Final Evaluation

Evaluate the optimized program:

In [ ]:
print("Evaluating optimized program...")
optimized_result = evaluate(optimized_program)

print(f"\n{'='*50}")
print(f"Baseline:  {baseline_result.score/100.:.1%}")
print(f"Optimized: {optimized_result.score/100.:.1%}")
print(f"Improvement: {(optimized_result.score - baseline_result.score)/100.:.1%}")
print(f"{'='*50}")

Evaluating optimized program...
Average Metric: 1.00 / 1 (100.0%):   1%|          | 1/150 [00:07<19:46,  7.97s/it]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content="[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Average Metric: 2.00 / 2 (100.0%):   1%|▏         | 2/150 [00:08<09:02,  3.66s/it]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Average Metric: 47.00 / 60 (78.3%):  39%|███▉      | 59/150 [00:35<01:14,  1.22it/s]

...

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{\n  "re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Average Metric: 94.00 / 150 (62.7%): : 151it [02:31,  1.00s/it]                       

2025/10/18 12:23:49 INFO dspy.evaluate.evaluate: Average Metric: 94 / 150 (62.7%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,Analyze and Reframe: - This is a number theory problem about posit...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,Problem type: geometry (coordinate/analytic geometry with reflecti...,588,✔️ [1]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,Analyze and Reframe: - This is a counting (combinatorics) problem....,16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"Analyze and Reframe: We need integer ordered pairs (x,y) with x,y ...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Problem type: combinatorics / divisibility rules. We must count 8-...,279,✔️ [1]



Baseline:  53.3%
Optimized: 62.7%
Improvement: 9.3%


## Optimization Insights

Examine the optimization process:

In [ ]:
if hasattr(optimized_program, 'apex_result'):
    result = optimized_program.apex_result
    
    print("Summary:")
    print(f"  Iterations: {len(result.iterations)}")
    print(f"  Candidates evaluated: {len(result.all_candidates)}")
    print(f"  Stop reason: {result.stopped_after}")
    print(f"  Best score: {result.best_candidate.overall_score:.4f}")
    
    print("\nIteration Progress:")
    for it in result.iterations:
        print(f"  Iteration {it.iteration}: {it.num_failures} failures, {len(it.hypotheses)} hypotheses, {len(it.candidates)} candidates")
    
    if result.best_candidate.hypothesis:
        h = result.best_candidate.hypothesis
        print(f"\nBest Hypothesis:")
        print(f"  Strategy: {h.strategy if hasattr(h, 'strategy') else 'N/A'}")
        print(f"  Impact Score: {h.impact_score if hasattr(h, 'impact_score') else 'N/A'}")

Summary:
  Iterations: 9
  Candidates evaluated: 19
  Stop reason: interrupted
  Best score: 0.6889

Iteration Progress:
  Iteration 1: 2 failures, 1 hypotheses, 2 candidates
  Iteration 2: 2 failures, 1 hypotheses, 2 candidates
  Iteration 3: 6 failures, 1 hypotheses, 2 candidates
  Iteration 4: 3 failures, 1 hypotheses, 2 candidates
  Iteration 5: 3 failures, 1 hypotheses, 2 candidates
  Iteration 6: 3 failures, 1 hypotheses, 2 candidates
  Iteration 7: 2 failures, 1 hypotheses, 2 candidates
  Iteration 8: 1 failures, 1 hypotheses, 2 candidates
  Iteration 9: 3 failures, 1 hypotheses, 2 candidates

Best Hypothesis:
  Strategy: Enhance the proven 4-step methodology by injecting specific, targeted advice for the most common failure patterns (combinatorics, geometry) into the `Decompose` and `Verify` steps. This includes adding rules for case analysis, precise definitions, and sufficiency checks. Simultaneously, add a strict output formatting rule to the `Verify` step to eliminate a sep

## Conclusion

APEX systematically optimizes prompts through:
1. Analyzing failures to understand root causes
2. Recognizing successful patterns
3. Generating data-driven hypotheses
4. Validating improvements empirically

Try adjusting the configuration parameters to explore different optimization strategies.